# Handle Error and Error Response

```{eval-rst}

When using the QC-Test Client, you may encounter various exceptions that indicate issues with your requests or the server's response. The following sections provide guidance on how to handle these exceptions and retrieve detailed error information.
QC-Test Client uses a custom exception hierarchy to represent different types of errors that may occur during the execution of requests. All exceptions inherit from :class:`QCTSSException <qctss_client.exceptions.QCTSSException>`, which itself inherits from Python's built-in :class:`Exception` class.

.. seealso::
    The diagram of the exception hierarchy can be found in `exceptions </apidoc/exceptions.html>`_.

.. hint::
    When no error occurs, the response from the server is returned as a :class:`JobResponse <qctss_client.models.JobResponse>` object, which is processed by the client in other modules. You can refer to the :class:`JobResponse <qctss_client.models.JobResponse>` documentation for more details on the response structure and its attributes.
```


In [ ]:
from qctss_client import QCTSSClient, JobStatus
from qctss_client.exceptions import (
    AuthenticationError,
    QCSetupConfigNotFoundError,
    QCSetupNotFoundError,
    ValidationError,
    JobClientError,
    WebSocketError,
    QCTSSTimeoutError,
    QCTSSException,
)

```{eval-rst}

The exceptions of ``qctss_client``, :class:`QCTSSException <qctss_client.exceptions.QCTSSException>`,  contain the some extra attributes, which can be used to get more information about the error.

- :attr:`http_status <qctss_client.exceptions.QCTSSException.http_status>`: The HTTP status code returned by the server.
- :attr:`error_code <qctss_client.exceptions.QCTSSException.error_code>`: The specific error code.
- :attr:`backend_message <qctss_client.exceptions.QCTSSException.backend_message>`: The message from the backend.
- :attr:`details <qctss_client.exceptions.QCTSSException.details>`: Additional details about the error. Usually, it's the response body returned by the server when error occurs. It can be a string or a dictionary, depending on the error.

```

`print_error_details` function in the following cell is the helper function to print the error details.


In [ ]:
def handle_update(status: JobStatus):
    print(f"Job {status.job_id}: {status.status}")


def print_error_details(reason: str, e: QCTSSException):
    print(f"Error ({reason}): {e}")
    print(f"-" * 50)
    print(f"HTTP Status: {e.http_status}")
    print(f"Error Code: {e.error_code}")
    print(f"Backend Message: {e.backend_message}")
    print(f"Details: {e.details}")

In [13]:
client = QCTSSClient()

try:
    # Submit job
    job = client.start_job(["setup1"], "quantum_sim")

    client.subscribe_job_updates(job.job_id, handle_update)

except AuthenticationError as e:
    print_error_details("Authentication failed", e)
except QCSetupConfigNotFoundError as e:
    print_error_details("QCSetup has no activated config", e)
except QCSetupNotFoundError as e:
    print_error_details("QCSetup not found", e)
except ValidationError as e:
    print_error_details("Invalid parameters", e)
except JobClientError as e:
    print_error_details("Job operation failed", e)
except WebSocketError as e:
    print_error_details("WebSocket error", e)
except QCTSSTimeoutError as e:
    print_error_details("Request timed out", e)
except QCTSSException as e:
    print_error_details("Any other error from this package", e)

finally:
    client.close()

HTTP 400 error: {"error":["Service 'quantum_sim' not found"]}


Error (Job operation failed): Job creation failed: ["Service 'quantum_sim' not found"] | HTTP "400" | Code: "CLIENT_ERROR" | Backend: "["Service 'quantum_sim' not found"]"
--------------------------------------------------
HTTP Status: 400
Error Code: CLIENT_ERROR
Backend Message: ["Service 'quantum_sim' not found"]
Details: {'status_code': 400, 'error': ["Service 'quantum_sim' not found"]}
